# 06 - AWS vs GCP Side-by-Side Cost and Performance Dashboard

This notebook combines the AWS and Google Cloud worksheets into one comparative view.

## Goals
- Compare Athena vs BigQuery for query-serving workloads
- Compare EMR vs Dataproc for batch distributed jobs
- Evaluate platform preference across multiple query-volume scenarios

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path.cwd().parent
else:
    REPO_ROOT = Path.cwd()

OUT_DIR = REPO_ROOT / 'notebooks' / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Output dir:', OUT_DIR)

## 1) Baseline Assumptions (Edit These)

Adjust these placeholders to your own observed cloud metrics.

In [ ]:
# Query engines
ATHENA_PRICE_PER_TB = 5.0
BIGQUERY_PRICE_PER_TB = 5.0

ATHENA_AVG_SCAN_GB = 0.25
BIGQUERY_AVG_PROCESSED_GB = 0.20

ATHENA_RUNTIME_SEC = 4
BIGQUERY_RUNTIME_SEC = 3

# Batch engines
EMR_HOURLY_USD = 0.576
DATAPROC_HOURLY_USD = 0.82

EMR_JOB_MINUTES = 25
DATAPROC_JOB_MINUTES = 25

# Scenario query volumes
SCENARIO_QUERIES_PER_DAY = {
    'low': 20,
    'medium': 150,
    'high': 600,
}

In [ ]:
def query_cost_usd(scan_gb: float, price_per_tb: float) -> float:
    return (scan_gb / 1024.0) * price_per_tb


def batch_cost_usd(hourly: float, minutes: float) -> float:
    return hourly * (minutes / 60.0)


athena_per_query = query_cost_usd(ATHENA_AVG_SCAN_GB, ATHENA_PRICE_PER_TB)
bq_per_query = query_cost_usd(BIGQUERY_AVG_PROCESSED_GB, BIGQUERY_PRICE_PER_TB)

emr_per_batch = batch_cost_usd(EMR_HOURLY_USD, EMR_JOB_MINUTES)
dataproc_per_batch = batch_cost_usd(DATAPROC_HOURLY_USD, DATAPROC_JOB_MINUTES)

base = pd.DataFrame([
    {'service': 'Athena', 'type': 'query', 'unit_cost_usd': athena_per_query, 'runtime_sec': ATHENA_RUNTIME_SEC},
    {'service': 'BigQuery', 'type': 'query', 'unit_cost_usd': bq_per_query, 'runtime_sec': BIGQUERY_RUNTIME_SEC},
    {'service': 'EMR', 'type': 'batch', 'unit_cost_usd': emr_per_batch, 'runtime_sec': EMR_JOB_MINUTES * 60},
    {'service': 'Dataproc', 'type': 'batch', 'unit_cost_usd': dataproc_per_batch, 'runtime_sec': DATAPROC_JOB_MINUTES * 60},
])

base.assign(unit_cost_usd=lambda d: d['unit_cost_usd'].round(6))

## 2) Scenario Matrix (Daily Cost)

Daily query cost is estimated for low/medium/high usage profiles. Batch cost assumes one run/day.

In [ ]:
rows = []
for scenario, qpd in SCENARIO_QUERIES_PER_DAY.items():
    rows.append({
        'scenario': scenario,
        'queries_per_day': qpd,
        'athena_daily_query_cost_usd': athena_per_query * qpd,
        'bigquery_daily_query_cost_usd': bq_per_query * qpd,
        'emr_daily_batch_cost_usd': emr_per_batch,
        'dataproc_daily_batch_cost_usd': dataproc_per_batch,
    })

scenario_df = pd.DataFrame(rows)
for col in scenario_df.columns:
    if col.endswith('_usd'):
        scenario_df[col] = scenario_df[col].round(5)

scenario_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Query serving comparison
qplot = scenario_df.set_index('scenario')[['athena_daily_query_cost_usd', 'bigquery_daily_query_cost_usd']]
qplot.plot(kind='bar', ax=axes[0], color=['#8E24AA', '#1E88E5'])
axes[0].set_title('Daily Query-Serving Cost')
axes[0].set_ylabel('USD/day')
axes[0].tick_params(axis='x', rotation=0)

# Batch comparison
bplot = scenario_df.set_index('scenario')[['emr_daily_batch_cost_usd', 'dataproc_daily_batch_cost_usd']]
bplot.plot(kind='bar', ax=axes[1], color=['#FB8C00', '#43A047'])
axes[1].set_title('Daily Batch Compute Cost (1 run/day)')
axes[1].set_ylabel('USD/day')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 3) Break-Even Calculations

How many daily queries make query-serving costs exceed one daily batch run cost?

In [ ]:
athena_vs_emr_break_even = np.inf if athena_per_query == 0 else emr_per_batch / athena_per_query
bq_vs_dataproc_break_even = np.inf if bq_per_query == 0 else dataproc_per_batch / bq_per_query

print(f'Athena queries/day to match one EMR run: {athena_vs_emr_break_even:.1f}')
print(f'BigQuery queries/day to match one Dataproc run: {bq_vs_dataproc_break_even:.1f}')

## 4) Decision Helper

Use this simple ruleset as a starting point, then override with real constraints.

In [ ]:
def recommend_stack(workload_type: str, latency_requirement: str = 'minutes') -> str:
    workload_type = workload_type.lower().strip()
    latency_requirement = latency_requirement.lower().strip()

    if workload_type in {'ad-hoc sql', 'interactive dashboard', 'bi'}:
        return 'Prefer serverless SQL (Athena or BigQuery), with partition-aware table design.'

    if workload_type in {'heavy etl', 'feature engineering', 'large spark transform'}:
        return 'Prefer distributed batch compute (EMR or Dataproc) for control and scale.'

    if workload_type in {'hybrid', 'near-real-time + batch'}:
        return 'Use mixed architecture: stream/query layer + scheduled batch reconciliation.'

    if latency_requirement in {'seconds', 'sub-minute'}:
        return 'Use low-latency serving/streaming path; do not rely on batch-only architecture.'

    return 'Run a pilot with both query-serving and batch paths; compare cost, SLA, and ops overhead.'


examples = [
    ('ad-hoc sql', 'seconds'),
    ('heavy etl', 'hours'),
    ('hybrid', 'seconds'),
]

for wt, lat in examples:
    print(f'{wt} | {lat} -> {recommend_stack(wt, lat)}')

In [ ]:
out_base = OUT_DIR / 'aws_gcp_tradeoff_baseline.csv'
out_scenarios = OUT_DIR / 'aws_gcp_tradeoff_scenarios.csv'

base.to_csv(out_base, index=False)
scenario_df.to_csv(out_scenarios, index=False)

print('Wrote:')
print('-', out_base)
print('-', out_scenarios)

## Reflection Prompts
1. Which platform pair is more cost-sensitive to query volume in your assumptions?
2. What changes most if you reduce average scan/processed GB per query by 50%?
3. Which operational factor (team skills, governance, debugging) outweighs pure cost in your context?
4. How would you justify a hybrid architecture to stakeholders?